# Configurable NLP Experimentation Workbench
### Hardware-Aware NLP Research, Rapid Prototyping & Gemastik Data Mining

An end-to-end, modular, hardware-aware NLP experimentation framework designed for:
1. **Gemastik Data Mining Preparation**: Strict competition pipeline, unlabelled test evaluation, ID alignment, and submission verification.
2. **Reproducible NLP Research**: Fold-local preprocessing, leak-free cross-validation, held-out test isolation, and explainable decision support.
3. **Hardware Efficiency**: Tailored for NVIDIA GeForce RTX 3050 (4GB/6GB VRAM) and CPU-only environments with dynamic batch sizing and mixed precision.
4. **Architecture Diversity**: 14 task categories × 3 executable approaches = 42 real model implementations integrated via the Architecture Registry.


## 00. Environment Setup & Package Verification
Verify that `nlp_workbench` and required machine learning dependencies are installed.


In [ ]:
import sys
from pathlib import Path

# Verify and import nlp_workbench package
try:
    import nlp_workbench
    print(f"nlp_workbench version: {nlp_workbench.__version__}")
except ImportError:
    # If running directly from git root without pip install -e .
    sys.path.insert(0, str(Path("src").resolve()))
    import nlp_workbench
    print(f"nlp_workbench version: {nlp_workbench.__version__} (loaded from src/)")


## 01. Configuration — Experiment Control Center
Configure the experimental setup in one centralized location.

### Supported Tasks:
`classification`, `regression`, `multilabel`, `ner`, `qa`, `summarization`, `translation`, `generation`, `similarity`, `retrieval`, `topic_modeling`, `pair_classification`, `zeroshot`.


In [ ]:
from nlp_workbench import WorkbenchConfig, get_default_config

# Initialize configuration
# To use your own dataset, set dataset_path="path/to/dataset.csv"
config = get_default_config(task="classification", language="indonesian")

config.data_path = "synthetic"     # Change to your dataset file path or "synthetic"
config.text_column = "text"
config.target_column = "target"
config.device = "auto"             # "auto", "cuda", or "cpu"
config.split_strategy = "stratified"
config.n_splits = 3
config.model_ids = ["classical_nb", "classical_logistic"]  # Offline-ready default
config.gemastik_mode = False       # Set to True when participating in Gemastik Data Mining

selected_models_str = config.model_ids if config.model_ids else "Auto-select top compatible"
print("Active Workbench Configuration:")
print(f"  Task:            {config.task}")
print(f"  Language:        {config.language}")
print(f"  Dataset Path:    {config.data_path}")
print(f"  Text Column(s):  {config.text_columns}")
print(f"  Target Column:   {config.target_column}")
print(f"  Target Device:   {config.device}")
print(f"  Selected Models: {selected_models_str}")
print(f"  CV Folds:        {config.n_splits}")
print(f"  Gemastik Mode:   {config.gemastik_mode}")


## 02. Hardware Profiling & VRAM Diagnostics
Inspects host hardware (CPU, CUDA, GPU model, VRAM limits) and calculates safe batch sizes and gradient accumulation steps for RTX 3050 (4GB) or CPU systems.


In [ ]:
from nlp_workbench import HardwareManager

# Inspect host hardware
hw = HardwareManager.inspect(config.device)
print(hw.summary())

# Hardware recommendations
rec = HardwareManager(config.device).recommend_batch_size()
print(f"\nHardware Manager Recommended Settings:")
print(f"  Batch Size:             {rec['batch_size']}")
print(f"  Grad Accumulation Steps:{rec['grad_accum_steps']}")


## 03. Dataset Ingestion & Schema Validation
Loads tabular data (CSV, TSV, Parquet, JSONL) or synthetic benchmark data with strict schema validation.
Never silently fabricates data if a real dataset path is specified.


In [ ]:
from nlp_workbench import ExperimentRunner, DataLoader, validate_dataset_schema

# Initialize canonical ExperimentRunner as the single source of truth
runner = ExperimentRunner(config)

# Execute staged preparation (ingestion, schema check, EDA profiling, 3-way split & leakage audit)
prepared = runner.prepare()

raw_df = prepared.raw_data.queries_df if hasattr(prepared.raw_data, "queries_df") else prepared.raw_data
print(f"Dataset Loaded Successfully: {raw_df.shape[0]} rows, {raw_df.shape[1]} columns")

is_valid, errors = validate_dataset_schema(prepared.raw_data, config)
if is_valid:
    print("Schema Validation Passed: Required columns exist and contain valid labels.")
else:
    print(f"Schema Warnings/Errors Detected: {errors}")

raw_df.head(5)


## 04 & 05. Exploratory Data Analysis & 5-Point Explainable Decision Support
Audits dataset quality and provides actionable 5-point decision support:
1. **What is the issue**
2. **Evidence observed**
3. **Why it matters**
4. **Recommended action**
5. **Potential drawbacks & controlling configuration variable**


In [ ]:
print("--- Column-by-Column Profiling ---")
summary_stats = prepared.eda_summary.get("column_profiles", getattr(prepared.audit_report, "summary_df", None))
print(summary_stats.to_string() if hasattr(summary_stats, "to_string") else summary_stats)

print("\n--- Actionable Decision Board ---")
decision_df = getattr(prepared.audit_report, "decision_board", None)
if decision_df is not None and hasattr(decision_df, "empty") and not decision_df.empty:
    display_cols = [c for c in ["Severity", "Finding", "Why It Matters", "Recommended Action", "Action Key"] if c in decision_df.columns]
    print(decision_df[display_cols].to_string(index=False))
else:
    print("No critical data quality issues identified. Dataset is clean.")


## 06 & 07. Preprocessing & Non-Destructive Cleaning
Recommends preprocessing configurations based on task, model family, and language.
Applies negation-preserving Indonesian stopword filtering and HTML/URL normalization.


In [ ]:
from nlp_workbench import recommend_preprocessing, TextCleaner

# Obtain automated recommendation
rec = recommend_preprocessing(task=config.task, model_family="classical", language=config.language)
print("Automated Preprocessing Recommendation:")
print(f"  Lowercase:              {rec.lowercase}")
print(f"  Remove HTML / URLs:     {rec.remove_html} / {rec.handle_urls}")
print(f"  Remove Stopwords:       {rec.remove_stopwords}")
print(f"  Preserve Negations:     {rec.preserve_negations}")
print(f"  N-Gram Range:           {rec.ngram_range}")
print(f"  Rationale:              {rec.rationale}")

# Instantiate TextCleaner
cleaner = TextCleaner(
    remove_html=config.accepted_decisions.get("enable_html_cleaning", True),
    handle_urls=config.accepted_decisions.get("handle_urls", True),
    lowercase=True,
    language=config.language,
)

# Test cleaning on sample text
sample_raw = "<p>Pelayanan rumah sakit ini TIDAK ramah sama sekali! Kunjungi https://rs-info.id</p>"
sample_cleaned = cleaner.clean_text(sample_raw)
print(f"\nSample Raw:     {sample_raw}")
print(f"Sample Cleaned: {sample_cleaned}")


## 08. Dataset Splitting & Strict Leakage Auditing
Splits the dataset into untouched Train, Validation, and Held-Out Test sets.
Performs rigorous data leakage auditing for:
- Duplicate text across splits
- Overlapping sample IDs
- Contaminated author / document groups


In [ ]:
tr_len = len(prepared.train_df)
va_len = len(prepared.val_df)
te_len = len(prepared.test_df)
tot = tr_len + va_len + te_len

print("Dataset Partitions Created:")
print(f"  Train Set:        {tr_len} samples ({tr_len / tot * 100:.1f}%)")
print(f"  Validation Set:   {va_len} samples ({va_len / tot * 100:.1f}%)")
print(f"  Held-Out Test Set:{te_len} samples ({te_len / tot * 100:.1f}%)")

print("\n--- Strict 3-Way Data Leakage Audit ---")
lr = prepared.leakage_report
tv_leak = lr["train_val"]["leakage_detected"]
tv_dups = lr["train_val"]["duplicate_text_count"]
tt_leak = lr["train_test"]["leakage_detected"]
tt_dups = lr["train_test"]["duplicate_text_count"]
vt_leak = lr["val_test"]["leakage_detected"]
vt_dups = lr["val_test"]["duplicate_text_count"]
all_leak = lr["leakage_detected"]

print(f"  Train/Val Leakage Detected: {tv_leak} (Duplicates: {tv_dups})")
print(f"  Train/Test Leakage Detected:{tt_leak} (Duplicates: {tt_dups})")
print(f"  Val/Test Leakage Detected:  {vt_leak} (Duplicates: {vt_dups})")
print(f"  Overall Leakage Detected:   {all_leak}")


## 09 & 10. Architecture Registry (14 Categories × 3 Models = 42 Approaches)
Every selectable model is backed by a callable execution adapter with lazy loading.
Inspect the models available for the active task.


In [ ]:
from nlp_workbench import registry

print(f"Total Models in Architecture Registry: {len(registry.models)}")

# Filter models suitable for current task and language
task_models = registry.list_models(task=config.task, language=config.language)
print(f"Models Compatible with '{config.task}' in '{config.language}': {len(task_models)}\n")

df_models = registry.to_dataframe(task=config.task, language=config.language)
print(df_models[["model_id", "category", "approach_name", "framework", "vram_tier"]].to_string(index=False))


## 11. Model Training & Fold-Local Cross-Validation
Executes cross-validation on the training set.
**Methodological Integrity**: Preprocessing (TF-IDF vectorizer, scalers) is fit strictly within each fold slice to guarantee zero feature leakage.


In [ ]:
# Execute canonical experiment lifecycle (training, cross-validation, evaluation, export & verification)
print("Starting Canonical Model Training & Validation via ExperimentRunner:")
result = runner.run(prepared)

print(f"\nExperiment execution complete. Evaluated {len(result.leaderboard)} candidate models.")


## 12. Unified Model Comparison & Leaderboard
Aggregates cross-validation metrics across all evaluated architectures into a clean comparison table.


In [ ]:
print("=== Validation / Cross-Validation Leaderboard ===")
print(result.leaderboard.to_string(index=False))


## 13. Final Model Selection & Held-Out Test Evaluation
**Scientific Rigor**:
1. The winning architecture is selected **strictly** using validation / CV scores.
2. The held-out test set is evaluated **exactly once** on the selected model to estimate true generalization error.


In [ ]:
print(f"Selected Winning Architecture: {result.best_model_id}")
print()
print("Held-Out Test Evaluation (Evaluated Exactly Once):")
for metric_name, val in result.test_metrics.items():
    if isinstance(val, (int, float)):
        print(f"  {metric_name}: {val:.4f}")
    elif isinstance(val, list) and len(val) < 10:
        print(f"  {metric_name}: {val}")


## 14. Error Analysis & Diagnostics
Examines misclassified samples and investigates prediction confidence.


In [ ]:
import numpy as np

test_df = result.test_df.queries_df if hasattr(result.test_df, "queries_df") else result.test_df
test_preds = result.best_adapter.predict(test_df)
test_eval = test_df.copy()
test_eval["prediction"] = test_preds

text_c = config.text_column or (config.text_columns[0] if config.text_columns else "text")
target_c = config.target_column

if config.task == "classification" and target_c in test_df.columns:
    test_eval["is_correct"] = (test_eval[target_c].astype(str) == test_eval["prediction"].astype(str))
    acc = test_eval["is_correct"].mean()
    print(f"Held-Out Test Accuracy: {acc * 100:.2f}%")
    misclassified = test_eval[~test_eval["is_correct"]]
    print(f"Misclassified Samples: {len(misclassified)} of {len(test_eval)}")
    if not misclassified.empty and text_c in misclassified.columns:
        sample_errors = misclassified[[text_c, target_c, "prediction"]].head(5)
        print("\nSample Misclassifications:")
        print(sample_errors.to_string(index=False))
elif config.task == "regression" and target_c in test_df.columns:
    residuals = np.abs(test_df[target_c].values - np.array(test_preds))
    print(f"Mean Absolute Error on Test: {np.mean(residuals):.4f}")
    print(f"Max Residual on Test: {np.max(residuals):.4f}")
else:
    print(f"Diagnostics completed successfully for task {config.task}.")


## 15. Complete Model Packaging & Export
Exports the winning adapter, complete with vectorizers, encoders, metadata, and preprocessing pipeline, into a self-contained artifact.


In [ ]:
from nlp_workbench import ModelExporter

print("Model Artifact Exported Successfully by ExperimentRunner:")
print(f"  Export Path: {result.exported_path}")
print(f"  Artifact Exists on Disk: {result.exported_path.exists()}")


## 16. Standalone Production Inference (`predict_new`)
Provides a single-line production inference function that accepts raw strings, lists of texts, or DataFrames and returns predictions and confidence scores.


In [ ]:
from nlp_workbench import predict_new, ModelExporter

# Load exported bundle for standalone inference
loaded_bundle = ModelExporter.load(result.exported_path)

sample_queries = [
    "Pemerintah mengumumkan kebijakan fiskal baru untuk mengatasi inflasi perbankan",
    "Tim dokter berhasil menyelesaikan uji klinis vaksin terbaru dengan aman",
    "Startup teknologi meluncurkan komputasi cloud berbasis kecerdasan buatan",
]

inference_result = predict_new(
    texts=sample_queries,
    adapter=loaded_bundle,
)

print("--- Standalone Production Inference Results ---")
print(inference_result.df[["input_text", "prediction", "confidence"]].to_string(index=False))


## 17. Gemastik Data Mining Competition Mode
When `gemastik_mode=True`, ingests the unlabelled competition test set, performs inference, strictly verifies row preservation and ID alignment, and outputs `submission.csv`.


In [ ]:
if config.gemastik_mode:
    print("=== Gemastik Data Mining Competition Results ===")
    summary = result.competition_summary
    sub_p = summary["submission_path"]
    r_cnt = summary["row_count"]
    cols = summary["columns"]
    prev = summary["sample_head"]
    print(f"  Submission File: {sub_p}")
    print(f"  Row Count:       {r_cnt}")
    print(f"  Columns:         {cols}")
    print(f"  Preview:         {prev}")
else:
    print("Gemastik competition mode is disabled (config.gemastik_mode = False).")
    print("To generate a verified competition submission:")
    print("  1. Set config.gemastik_mode = True")
    print("  2. Provide config.competition_test_path = 'path/to/test.csv'")
    print("  3. Run runner.run(prepared)")
    print("The held-out evaluation test set remains strictly isolated and is never reused.")


## 18. Experiment Summary & Reproducibility Audit
Generates a comprehensive audit record of the experiment including git commit, dependencies, hyperparameters, and environment settings.


In [ ]:
import json

print("=== Experiment Reproducibility Audit ===")
print(json.dumps(result.reproducibility_report, indent=2))
